In [1]:
!pip install catboost -q

import pandas as pd
import numpy as np

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

train.head()

Train shape: (77299, 11)
Test shape: (41778, 10)


,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


In [3]:
train.isnull().sum()

Index               0
geohash             0
day                 0
timestamp           0
demand              0
RoadType          600
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      2495
Weather           797
dtype: int64

In [7]:
for col in ["RoadType", "Weather"]:
    train[col] = train[col].fillna(train[col].mode()[0])
    test[col] = test[col].fillna(test[col].mode()[0])

train["Temperature"] = train["Temperature"].fillna(train["Temperature"].median())
test["Temperature"] = test["Temperature"].fillna(test["Temperature"].median())

In [9]:
train.isnull().sum()

Index            0
geohash          0
day              0
timestamp        0
demand           0
RoadType         0
NumberofLanes    0
LargeVehicles    0
Landmarks        0
Temperature      0
Weather          0
dtype: int64

In [11]:
def create_features(df):

    temp = df["timestamp"].str.split(":", expand=True)

    df["hour"] = temp[0].astype(int)
    df["minute"] = temp[1].astype(int)

    df["rush_hour"] = (
        ((df["hour"] >= 7) & (df["hour"] <= 10))
        |
        ((df["hour"] >= 17) & (df["hour"] <= 20))
    ).astype(int)

    df["is_weekend"] = (df["day"] % 7 >= 5).astype(int)

    return df

train = create_features(train)
test = create_features(test)

train.head()

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,hour,minute,rush_hour,is_weekend
0,0,qp02z1,48,0:0,0.048804,Residential,1,Not Allowed,No,16.382587,Sunny,0,0,0,1
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny,0,0,0,1
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny,0,0,0,1
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,16.382587,Rainy,0,0,0,1
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy,0,0,0,1


In [13]:
from catboost import CatBoostRegressor

FEATURES = [
    "geohash",
    "day",
    "RoadType",
    "NumberofLanes",
    "LargeVehicles",
    "Landmarks",
    "Temperature",
    "Weather",
    "hour",
    "minute",
    "rush_hour",
    "is_weekend"
]

TARGET = "demand"

X_train = train[FEATURES]
y_train = train[TARGET]

X_test = test[FEATURES]

cat_features = [
    "geohash",
    "RoadType",
    "LargeVehicles",
    "Landmarks",
    "Weather"
]

model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function="RMSE",
    random_seed=42,
    verbose=100
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_features
)

0:	learn: 0.1369185	total: 74.4ms	remaining: 1m 14s
100:	learn: 0.0430406	total: 2.29s	remaining: 20.4s
200:	learn: 0.0396958	total: 4.45s	remaining: 17.7s
300:	learn: 0.0380246	total: 6.77s	remaining: 15.7s
400:	learn: 0.0365708	total: 8.99s	remaining: 13.4s
500:	learn: 0.0352000	total: 11.7s	remaining: 11.6s
600:	learn: 0.0342819	total: 14s	remaining: 9.31s
700:	learn: 0.0333771	total: 17.1s	remaining: 7.28s
800:	learn: 0.0326868	total: 19.6s	remaining: 4.87s
900:	learn: 0.0321452	total: 21.8s	remaining: 2.4s
999:	learn: 0.0316431	total: 23.8s	remaining: 0us


CatBoostRegressor(depth=8, iterations=1000, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=100)

In [15]:
preds = model.predict(X_test)

print(preds[:10])

[0.04117076 0.02353006 0.00580721 0.02857058 0.05465348 0.01047035
 0.02442965 0.06131131 0.02845646 0.04743695]


In [20]:
print("Train Shape:", train.shape)
print("Test Shape:", test.shape)

print("Predictions Length:", len(preds))

sample = pd.read_csv("sample_submission.csv")
print("Sample Shape:", sample.shape)

print(sample.head())

Train Shape: (77299, 15)
Test Shape: (41778, 14)
Predictions Length: 41778
Sample Shape: (5, 2)
   Index    demand
0      0  0.090768
1      1  0.089885
2      2  0.007037
3      3  0.079087
4      4  0.054636


In [22]:
print(type(preds))
print(preds.shape)

<class 'numpy.ndarray'>
(41778,)


In [24]:
submission = pd.DataFrame({
    "Index": test["Index"],
    "demand": preds
})

submission.to_csv("submission.csv", index=False)

print(submission.shape)
submission.head()

(41778, 2)


,Index,demand
0,0,0.041171
1,1,0.023530
2,2,0.005807
3,3,0.028571
4,4,0.054653


In [26]:
submission.shape

(41778, 2)

In [30]:
submission.to_csv("submission.csv", index=False)